# Número de casos por Ano Diagnóstico, Escolaridade, Raça e Cor e Faixa Etária
## Resultados Parciais da Análise Exploratória dos Dados 


In [ ]:
# Bibliotecas padrão
import json
import re
import unicodedata
import warnings
from pathlib import Path

# Bibliotecas de terceiros
import geobr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# Configurações
warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"

In [ ]:
CSV_CASOS_ANO_DIAGNOSTICO  = '/home/anna/tcc/Tratamento/Casos/incidencia _ano_diagnostico.csv'
CSV_CASOS_ESCOLARIDADE     = '/home/anna/tcc/Tratamento/Casos/incidencia _escolaridade.csv'
CSV_CASOS_RACA_COR         = '/home/anna/tcc/Tratamento/Casos/incidencia _raca_cor.csv'
CSV_CASOS_FAIXA_ETARIA     = '/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv'

CORRECAO_GRAFIA = {
    'indigena':                     'Indígena',
    'parda':                        'Parda',
    'preta':                        'Preta',
    'branca':                       'Branca',
    'amarela':                      'Amarela',
    'ignorado':                     'Ignorado',
    'ignorada':                     'Ignorado',
    'sem informacao':               'Sem informação',
    'nenhuma':                      'Nenhuma',
    'fundamental incompleto':       'Fundamental incompleto',
    'fundamental completo':         'Fundamental completo',
    'medio incompleto':             'Médio incompleto',
    'medio completo':               'Médio completo',
    'nivel medio':                  'Nível médio',
    'nivel medio incompleto':       'Nível médio incompleto',
    'nivel medio completo':         'Nível médio completo',
    'nivel superior incompleto':    'Nível superior incompleto',
    'nivel superior completo':      'Nível superior completo',
    'nao se aplica':                'Não se aplica',
    'nao informado':                'Não informado',
    'nao informada':                'Não informado',
}

## Gráficos de Barras e Distribuição

In [ ]:

MUNICIPIO             = None
CODIGO_MUNICIPIO      = None
REGIONAL_SAUDE        = None
ANOS_RANGE_CASOS      = (1985, 2023)
TOP_N_CATEGORIAS      = 25
COR_BARRAS            = "#6ca8e5"
COR_PONTOS            = "#d73f3f"
COR_TEXTO_BARRA       = "#1a1a1a"
LARGURA_GRAFICO       = 2300
ALTURA_GRAFICO_CASOS  = 1700
X_MIN                 = 0
DIVISOR_MEDIA_ANUAL_CASOS       = 10
ANO_REFERENCIA_POPULACAO_CASOS  = 2022
ANO_MALHA_MUNICIPIOS_MG         = 2020
FILTRO_FAIXA_ETARIA             = None
ORDENAR_TABELA_POR              = 'valor_absoluto_casos'
FAIXAS_ETARIAS_ORDEM = ['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70 ou mais']

def _corrigir_grafia(texto: str) -> str:
    chave = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode('ascii').lower().strip()
    return CORRECAO_GRAFIA.get(chave, str(texto).strip())

# ── Utilitários ────────────────────────────────────────────────────────────────
def _norm_txt(txt: str) -> str:
    return unicodedata.normalize('NFKD', str(txt)).encode('ascii', 'ignore').decode('ascii').lower().strip()

def _norm_col(col: str) -> str:
    return re.sub(r'\s+', '_', _norm_txt(col))

def _find_first_col(df: pd.DataFrame, candidates_norm: set):
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None

def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols_drop = [c for c in df.columns if _norm_col(c).startswith('unnamed')]
    return df.drop(columns=cols_drop) if cols_drop else df.copy()

def _fmt_int(n):
    try:
        return f'{int(n):,}'.replace(',', '.')
    except Exception:
        return str(n)

def normalizar_faixa_etaria(categoria):
    cat = _norm_txt(categoria)
    if cat in {'70-79','70 a 79','80+','80 ou mais','80 ou mais.','70 ou mais'}:
        return '70 ou mais'
    return str(categoria).strip()

def _normalizar_codigo_municipio(serie: pd.Series) -> pd.Series:
    return (pd.to_numeric(serie, errors='coerce')
              .astype('Int64').astype(str)
              .str.replace('<NA>', '', regex=False)
              .str.zfill(7))

def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    out = _drop_index_cols(df)
    col_mun = _find_first_col(out, {'municipios','municipio','cidades','cidade'})
    col_cod = _find_first_col(out, {'codigo','cod'})
    col_reg = _find_first_col(out, {'regional_de_saude','regional_saude','regional'})
    if MUNICIPIO is not None and col_mun:
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]
    if CODIGO_MUNICIPIO is not None and col_cod:
        out = out[pd.to_numeric(out[col_cod], errors='coerce').astype('Int64') == int(CODIGO_MUNICIPIO)]
    if REGIONAL_SAUDE is not None and col_reg:
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]
    return out

def _year_cols(df):
    cols = []
    for c in df.columns:
        try:
            y = int(c)
            if 1900 <= y <= 2100:
                cols.append(c)
        except:
            pass
    return sorted(cols)

def preparar_ano_diagnostico(df):
    ycols = _year_cols(df)
    s = df[ycols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0)
    s = s[s > 0]
    return (pd.DataFrame({'ano': s.index.astype(int), 'categoria': s.index.astype(str), 'total': s.values})
              .sort_values('ano').reset_index(drop=True))

def preparar_long_wide(df):
    rows = []
    for c in df.columns:
        txt = str(c)
        if '_' not in txt:
            continue
        ano_str, categoria = txt.rsplit('_', 1)
        try:
            ano = int(ano_str)
        except:
            continue
        categoria = normalizar_faixa_etaria(categoria)
        total = pd.to_numeric(df[c], errors='coerce').fillna(0).sum()
        if total > 0:
            rows.append({'ano': ano, 'categoria': categoria, 'total': int(total)})
    out = pd.DataFrame(rows)
    return out.sort_values(['ano', 'categoria']).reset_index(drop=True)

def build_por_categoria_por_ano(df_long, anos_range=None, top_n=None):
    d = df_long.copy()
    d['ano']   = pd.to_numeric(d['ano'],   errors='coerce').astype('Int64')
    d['total'] = pd.to_numeric(d['total'], errors='coerce')
    d = d.dropna(subset=['ano', 'categoria', 'total'])
    d['ano'] = d['ano'].astype(int)
    d = d[d['total'] > 0]
    if anos_range is not None:
        ai, af = map(int, anos_range)
        d = d[(d['ano'] >= ai) & (d['ano'] <= af)]
    por_categoria = d.groupby('categoria')['total'].sum()
    if top_n is not None:
        por_categoria = por_categoria.sort_values(ascending=False).head(int(top_n)).sort_values()
    por_ano = d.groupby('ano')['total'].sum().sort_index()
    return por_categoria, por_ano

def _carregar_e_preparar_casos(caminho_csv, nome, modo='wide', top_n=None, anos_range=ANOS_RANGE_CASOS):
    df   = aplicar_filtros(pd.read_csv(caminho_csv))
    print(f'Linhas selecionadas ({nome}): {len(df)}')
    long = preparar_ano_diagnostico(df) if modo == 'ano_cols' else preparar_long_wide(df)
    return build_por_categoria_por_ano(long, anos_range=anos_range, top_n=top_n)

print('=== CONFIG / FILTRO ===')
print('MUNICIPIO:',        MUNICIPIO)
print('CODIGO_MUNICIPIO:', CODIGO_MUNICIPIO)
print('REGIONAL_SAUDE:',   REGIONAL_SAUDE)

# ── Parâmetros de eixo ─────────────────────────────────────────────────────────

FOLGA_ROTULOS_PONTOS = 0.18
_LIMIAR_FORA = 0.10

def _round_up_nice(v):
    v = float(v)
    if v <= 0:
        return 1000
    step = 200 if v <= 2000 else 500 if v <= 10000 else 1000
    return int(((v + step - 1) // step) * step)

def _build_ticks(xmax, n_ticks_max=10):
    raw_step  = xmax / n_ticks_max
    magnitude = 10 ** (len(str(int(raw_step))) - 1)
    step      = max(magnitude, round(raw_step / magnitude) * magnitude)
    return list(range(0, int(xmax) + step, step))

def _resolve_xmax(vals, fixo=None, folga=0.45, extra=0):
    d    = max(vals) if vals else 0
    r    = _round_up_nice(d * (1 + folga))
    base = r if fixo is None else max(int(fixo), r)
    return base + extra

# ── Plot: ano diagnóstico ──────────────────────────────────────────────────────
def plot_ano(por_ano, titulo, xmax_fixo=8000):
    s    = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    s    = s[s > 0]
    anos = s.index.astype(int).tolist()
    vals = s.values.astype(int).tolist()

    xmax  = 8000
    ticks = _build_ticks(xmax, n_ticks_max=8)

    text_pos = 'outside'

    fig = go.Figure(go.Bar(
        x=[str(a) for a in anos],
        y=vals,
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in vals],
        textposition=text_pos,
        textangle=0,
        insidetextanchor='end',
        textfont=dict(size=16, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
    ))
    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        margin=dict(l=100, r=200, t=90, b=100),
        xaxis=dict(
            title='Ano do diagnóstico',
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=0,
            tickmode='array',
            tickvals=[str(a) for a in anos],
            ticktext=[str(a) for a in anos],
            showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True,
        ),
        yaxis=dict(
            title='Número de casos',
            title_font=dict(size=28),
            tickfont=dict(size=16),
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax],
            showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True,
        ),
    )
    fig.show()

# ── Plot: duplo ────────────────────────────────────────────────────────────────
def plot_duplo(por_categoria, por_ano, titulo, ytitle, xmax_fixo):
    c = pd.to_numeric(por_categoria, errors='coerce').fillna(0)
    c = c[c > 0]
    a = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    a = a[a > 0]

    is_faixa = _norm_txt(ytitle) == 'faixa etaria'
    if is_faixa:
        ordem = [f for f in FAIXAS_ETARIAS_ORDEM if f in c.index.astype(str).tolist()]
        c = c.reindex(ordem)

    cats = [_corrigir_grafia(cat) for cat in c.index.astype(str).tolist()]
    v1   = c.values.astype(int).tolist()
    anos = a.index.astype(int).tolist()
    v2   = a.values.astype(int).tolist()

    # xmax da área de plotagem 
    xmax_plot = int(xmax_fixo) if xmax_fixo is not None else _resolve_xmax(v1, None, folga=0.30)
    
    # range do eixo X
    xmax_range = int(xmax_plot * (1 + FOLGA_ROTULOS_PONTOS))
    
    ticks = _build_ticks(xmax_plot, n_ticks_max=10) 

    ano_ini, ano_fim = map(int, ANOS_RANGE_CASOS)
    anos_eixo        = list(range(ano_ini, ano_fim + 1))

    fig = go.Figure()

    fig.add_trace(go.Bar(
        name='Total por categoria',
        x=v1,
        y=cats,
        orientation='h',
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in v1],
        textposition='outside',
        textfont=dict(size=20, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
        yaxis='y',
        xaxis='x',
    ))

    fig.add_trace(go.Scatter(
        name='Total por ano',
        x=v2,
        y=anos,
        mode='markers+text',
        marker=dict(color=COR_PONTOS, size=10),
        text=[_fmt_int(v) for v in v2],
        textfont=dict(color=COR_PONTOS, size=20),
        textposition='middle right',
        cliponaxis=False,
        yaxis='y2',
        xaxis='x',
    ))

    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',

        margin=dict(l=170, r=500, t=95, b=100),
        xaxis=dict(
            title='Número de casos',
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax_range],  
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=0,
        ),
        yaxis=dict(
            title=ytitle,
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            categoryorder='array',
            categoryarray=cats,
            automargin=True,
            title_font=dict(size=32),
            tickfont=dict(size=22),
        ),
        yaxis2=dict(
            title='Ano',
            overlaying='y',
            side='right',
            tickmode='array',
            tickvals=anos_eixo,
            range=[ano_ini - 2, ano_fim + 2],
            showgrid=False,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
        ),
    )
    fig.show()

# ── Carregar e plotar ──────────────────────────────────────────────────────────
graficos_casos = []

por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_ANO_DIAGNOSTICO, 'Casos - Ano Diagnóstico',
    modo='ano_cols', top_n=None, anos_range=None,
)
por_ano_ref = por_ano[(por_ano.index >= ANOS_RANGE_CASOS[0]) & (por_ano.index <= ANOS_RANGE_CASOS[1])]
graficos_casos.append({
    'tipo': 'ano', 'titulo': 'Casos - Ano Diagnóstico',
    'ylabel': 'Ano do diagnóstico', 'por_categoria': por_cat,
    'por_ano': por_ano_ref, 'x_max': 8000,
})

for nome, csv, ylabel, xmax in [
    ('Casos - Escolaridade', CSV_CASOS_ESCOLARIDADE, 'Escolaridade', 37000), 
    ('Casos - Raça/Cor',     CSV_CASOS_RACA_COR,     'Raça/Cor',     55000), 
    ('Casos - Faixa Etária', CSV_CASOS_FAIXA_ETARIA, 'Faixa etária', 33000),
]:
    c, a = _carregar_e_preparar_casos(csv, nome, modo='wide',
                                      top_n=TOP_N_CATEGORIAS,
                                      anos_range=ANOS_RANGE_CASOS)
    graficos_casos.append({
        'tipo': 'duplo', 'titulo': nome,
        'ylabel': ylabel, 'por_categoria': c,
        'por_ano': a, 'x_max': xmax,
    })

print(f'Casos: {len(graficos_casos)} gráficos preparados')

for g in graficos_casos:
    if g['tipo'] == 'ano':
        plot_ano(g['por_ano'], f"{g['titulo']} (total por ano)", g['x_max'])
    else:
        plot_duplo(g['por_categoria'], g['por_ano'], g['titulo'], g['ylabel'], g['x_max'])

print('Gráficos de casos plotados')

In [ ]:
# ── Plot: só barras horizontais (sem pontos de ano) ───────────────────────────
def plot_so_barras(por_categoria, titulo, ytitle, xmax_fixo):
    c = pd.to_numeric(por_categoria, errors='coerce').fillna(0)
    c = c[c > 0]

    is_faixa = _norm_txt(ytitle) == 'faixa etaria'
    if is_faixa:
        ordem = [f for f in FAIXAS_ETARIAS_ORDEM if f in c.index.astype(str).tolist()]
        c = c.reindex(ordem)

    cats = [_corrigir_grafia(cat) for cat in c.index.astype(str).tolist()]
    v1   = c.values.astype(int).tolist()

    xmax_plot  = int(xmax_fixo)
    xmax_range = int(xmax_plot * (1 + FOLGA_ROTULOS_PONTOS))
    ticks      = _build_ticks(xmax_plot, n_ticks_max=10)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=v1,
        y=cats,
        orientation='h',
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in v1],
        textposition='outside',
        textfont=dict(size=20, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
    ))

    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        margin=dict(l=170, r=200, t=95, b=100),
        xaxis=dict(
            title='Número de casos',
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax_range],
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=0,
        ),
        yaxis=dict(
            title=ytitle,
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            categoryorder='array',
            categoryarray=cats,
            automargin=True,
            title_font=dict(size=32),
            tickfont=dict(size=22),
        ),
    )
    fig.show()

# ── Gerar gráficos só de barras ───────────────────────────────────────────────
for nome, csv, ylabel, xmax in [
    ('Casos - Escolaridade', CSV_CASOS_ESCOLARIDADE, 'Escolaridade', 37000),
    ('Casos - Raça/Cor',     CSV_CASOS_RACA_COR,     'Raça/Cor',     55000),
    ('Casos - Faixa Etária', CSV_CASOS_FAIXA_ETARIA, 'Faixa etária', 33000),
]:
    c, _ = _carregar_e_preparar_casos(csv, nome, modo='wide',
                                      top_n=TOP_N_CATEGORIAS,
                                      anos_range=ANOS_RANGE_CASOS)
    plot_so_barras(c, nome, ylabel, xmax)

print('Gráficos de barras plotados')

## Mapas de Minas Gerais por Município e Regional de Saúde

In [ ]:
ANO_INI_MAPA     = 1985
ANO_FIM_MAPA     = 2023
COR_SEM_CASOS    = "#6ca8e5"
FAIXAS_MUNICIPIOS = [100, 300, 600, 1000, 3000]
FAIXAS_REGIONAIS = [500, 1000, 2500, 5000, 10000]
PALETA_MAPA      = ["#9c2328", "#d20000", "#ff7f50", "#ee9c21", "#fec44f", "#ffffd4"]

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def normalizar_texto(valor) -> str:
    if pd.isna(valor):
        return ""
    txt = str(valor).strip().lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("utf-8")
    txt = re.sub(r"\s+", " ", txt)
    return txt


def centro(gdf):
    b = gdf.total_bounds
    return (b[1] + b[3]) / 2, (b[0] + b[2]) / 2


def categoria_casos(n, faixas, cor_sem_casos):
    """Retorna o rótulo de categoria para um total de casos."""
    if n == 0:
        return "Sem casos"
    prev = 0
    for f in faixas:
        if n <= f:
            label = f"1–{f}" if prev == 0 else f"{prev + 1}–{f}"
            return label
        prev = f
    return f"> {faixas[-1]}"


# ── Localização dos arquivos ──────────────────────────────────────────────────
candidatos_base = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
path_casos     = None
path_municipios = None

for base in candidatos_base:
    pc = base / "Tratamento" / "Casos" / "incidencia _faixa_etaria.csv"
    pm = base / "Tratamento" / "Municípios" / "tabela_municipios_regionais.csv"
    if pc.exists() and pm.exists():
        path_casos      = pc
        path_municipios = pm
        break

if path_casos is None or path_municipios is None:
    raise FileNotFoundError(
        "Não foi possível localizar incidencia _faixa_etaria.csv e/ou "
        "tabela_municipios_regionais.csv"
    )

# ── Leitura ───────────────────────────────────────────────────────────────────
df_casos   = pd.read_csv(path_casos)
df_mun_reg = pd.read_csv(path_municipios)
muni_mg    = geobr.read_municipality(code_muni="MG", year=2020, simplified=True).to_crs(4326)

df_casos.columns   = [c.strip() for c in df_casos.columns]
df_mun_reg.columns = [c.strip() for c in df_mun_reg.columns]

# ── Soma acumulada por município (todas as faixas etárias, período selecionado) ─
anos_validos = [str(a) for a in range(ANO_INI_MAPA, ANO_FIM_MAPA + 1)]
colunas_casos = [
    c for c in df_casos.columns
    if "_" in c and c.split("_")[0] in anos_validos
]

df_casos["CODIGO_IBGE"] = pd.to_numeric(df_casos["CÓDIGO"], errors="coerce").astype("Int64")
df_casos["total_casos"] = (
    df_casos[colunas_casos]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .sum(axis=1)
    .astype(int)
)

# ── Categorias para a legenda ─────────────────────────────────────────────────
ordem_categorias = (
    ["Sem casos"]
    + [
        f"1–{FAIXAS_MUNICIPIOS[0]}",
        *[
            f"{FAIXAS_MUNICIPIOS[i] + 1}–{FAIXAS_MUNICIPIOS[i+1]}"
            for i in range(len(FAIXAS_MUNICIPIOS) - 1)
        ],
        f"> {FAIXAS_MUNICIPIOS[-1]}",
    ]
)

df_casos["categoria_casos"] = df_casos["total_casos"].apply(
    lambda n: categoria_casos(n, FAIXAS_MUNICIPIOS, COR_SEM_CASOS)
)

cores_categorias = {"Sem casos": COR_SEM_CASOS}
for label, cor in zip(reversed(ordem_categorias[1:]), PALETA_MAPA):
    cores_categorias[label] = cor

# ── Merge com geometria dos municípios ────────────────────────────────────────
gdf_municipios = muni_mg.merge(
    df_casos[["CODIGO_IBGE", "MUNICÍPIOS", "total_casos", "categoria_casos"]],
    left_on="code_muni",
    right_on="CODIGO_IBGE",
    how="left",
)
gdf_municipios["total_casos"]     = gdf_municipios["total_casos"].fillna(0).astype(int)
gdf_municipios["categoria_casos"] = gdf_municipios["categoria_casos"].fillna("Sem casos")
gdf_municipios["categoria_casos"] = pd.Categorical(
    gdf_municipios["categoria_casos"],
    categories=ordem_categorias,
    ordered=True,
)

gdf_municipios_plot       = gdf_municipios.reset_index(drop=True).copy()
gdf_municipios_plot["id"] = gdf_municipios_plot.index.astype(str)

# ── Mapa por município ────────────────────────────────────────────────────────
lat_c, lon_c = centro(gdf_municipios_plot)

fig_municipios = px.choropleth_map(
    gdf_municipios_plot,
    geojson=json.loads(gdf_municipios_plot.to_json()),
    locations="id",
    color="categoria_casos",
    color_discrete_map=cores_categorias,
    category_orders={"categoria_casos": ordem_categorias},
    title=f"Casos Acumulados de Câncer de Mama por Município em Minas Gerais ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="name_muni",
    hover_data={
        "total_casos":     True,
        "categoria_casos": False,
        "id":              False,
    },
    labels={"total_casos": "Total de casos"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5,
    opacity=0.95,
)
fig_municipios.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=760,
    legend_title_text="Nº de casos",
)
fig_municipios.show()

# ── Associação município → regional (mesma lógica do mapa de hospitais) ───────
df_mun_reg["codigo_ibge"]           = pd.to_numeric(df_mun_reg["codigo_ibge"], errors="coerce").astype("Int64")
df_mun_reg["municipio_normalizado"] = df_mun_reg["municipio_normalizado"].fillna("").map(normalizar_texto)
df_mun_reg["regional_original"]     = df_mun_reg["regional_original"].fillna("Nenhum")

df_casos["MUNICIPIO_NORM"] = df_casos["MUNICÍPIOS"].map(normalizar_texto)

mapa_regional_por_nome = dict(
    df_mun_reg[["municipio_normalizado", "regional_original"]]
    .drop_duplicates(subset=["municipio_normalizado"])
    .values
)

base_casos = df_casos.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="CODIGO_IBGE",
    right_on="codigo_ibge",
    how="left",
)
base_casos["regional_assoc"] = base_casos["regional_original"]
mask = base_casos["regional_assoc"].isna() | base_casos["regional_assoc"].eq("")
base_casos.loc[mask, "regional_assoc"] = (
    base_casos.loc[mask, "MUNICIPIO_NORM"].map(mapa_regional_por_nome)
)
base_casos["regional_assoc"] = base_casos["regional_assoc"].fillna("Nenhum")

# ── Soma de casos por regional ────────────────────────────────────────────────
combo_regional = (
    base_casos[base_casos["regional_assoc"] != "Nenhum"]
    .groupby("regional_assoc", as_index=False)
    .agg(total_casos=("total_casos", "sum"))
)
combo_regional["total_casos"] = combo_regional["total_casos"].astype(int)

combo_regional["categoria_casos"] = combo_regional["total_casos"].apply(
    lambda n: categoria_casos(n, FAIXAS_REGIONAIS, COR_SEM_CASOS)
)

ordem_categorias_reg = (
    ["Sem casos"]
    + [
        f"1–{FAIXAS_REGIONAIS[0]}",
        *[
            f"{FAIXAS_REGIONAIS[i] + 1}–{FAIXAS_REGIONAIS[i+1]}"
            for i in range(len(FAIXAS_REGIONAIS) - 1)
        ],
        f"> {FAIXAS_REGIONAIS[-1]}",
    ]
)

cores_categorias_reg = {"Sem casos": COR_SEM_CASOS}
for label, cor in zip(reversed(ordem_categorias_reg[1:]), PALETA_MAPA):
    cores_categorias_reg[label] = cor

# ── Geometria das regionais (dissolve) ────────────────────────────────────────
gdf_mg_reg = muni_mg.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="code_muni",
    right_on="codigo_ibge",
    how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

gdf_regionais = (
    gdf_mg_reg[gdf_mg_reg["regional_original"] != "Nenhum"][["regional_original", "geometry"]]
    .dissolve(by="regional_original", as_index=False)
)

gdf_regionais = gdf_regionais.merge(
    combo_regional,
    left_on="regional_original",
    right_on="regional_assoc",
    how="left",
)
gdf_regionais["total_casos"]     = gdf_regionais["total_casos"].fillna(0).astype(int)
gdf_regionais["categoria_casos"] = gdf_regionais["categoria_casos"].fillna("Sem casos")
gdf_regionais["categoria_casos"] = pd.Categorical(
    gdf_regionais["categoria_casos"],
    categories=ordem_categorias_reg,
    ordered=True,
)

gdf_regionais_plot       = gdf_regionais.reset_index(drop=True).copy()
gdf_regionais_plot["id"] = gdf_regionais_plot.index.astype(str)

# ── Mapa por regional ─────────────────────────────────────────────────────────
lat_c, lon_c = centro(gdf_regionais_plot)

fig_regionais = px.choropleth_map(
    gdf_regionais_plot,
    geojson=json.loads(gdf_regionais_plot.to_json()),
    locations="id",
    color="categoria_casos",
    color_discrete_map=cores_categorias_reg,
    category_orders={"categoria_casos": ordem_categorias_reg},
    title=f"Casos Acumulados de Câncer de Mama por Regional de Saúde em Minas Gerais ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="regional_original",
    hover_data={
        "total_casos":     True,
        "categoria_casos": False,
        "id":              False,
    },
    labels={"total_casos": "Total de casos"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5,
    opacity=0.95,
)
fig_regionais.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=760,
    legend_title_text="Nº de casos",
)
fig_regionais.show()


## TABELAS: Distribuição de casos por Faixa Etária, Escolaridade, Raça/Cor e Ano


In [ ]:


def _build_tabela(serie: pd.Series, col_categoria: str) -> pd.DataFrame:

    total = int(serie.sum())
    df = (
        serie
        .rename_axis(col_categoria)
        .reset_index(name="Valor Absoluto")
    )
    df["Valor Absoluto"] = df["Valor Absoluto"].astype(int)
    df["Percentual (%)"] = (df["Valor Absoluto"] / total * 100).map(
        lambda x: f"{x:,.2f}%".replace(",", "X").replace(".", ",").replace("X", ".")
    )
    df["Valor Absoluto_fmt"] = df["Valor Absoluto"].map(
        lambda x: f"{x:,}".replace(",", ".")
    )

    # Linha de total
    total_row = pd.DataFrame([{
        col_categoria:      "Total de Casos",
        "Valor Absoluto":   total,
        "Valor Absoluto_fmt": f"{total:,}".replace(",", "."),
        "Percentual (%)":   "100,00%",
    }])
    df = pd.concat([df, total_row], ignore_index=True)

    return df[[col_categoria, "Valor Absoluto_fmt", "Percentual (%)"]].rename(
        columns={"Valor Absoluto_fmt": "Valor Absoluto"}
    )


def _exibir_tabela(df: pd.DataFrame, caption: str) -> None:

    n = len(df)                     
    styles = [
        {"selector": "caption",
         "props": [("caption-side", "top"),
                   ("font-weight", "bold"),
                   ("font-size", "13px"),
                   ("text-align", "center"),
                   ("padding-bottom", "6px")]},
        {"selector": "th",
         "props": [("border", "1px solid #555"),
                   ("padding", "5px 10px"),
                   ("text-align", "center"),
                   ("background-color", "#f0f0f0")]},
        {"selector": "td",
         "props": [("border", "1px solid #aaa"),
                   ("padding", "4px 10px"),
                   ("text-align", "center")]},
    ]

    styled = (
        df.style
        .set_caption(caption)
        .set_table_styles(styles)
        .apply(
            lambda row: [
                "font-weight: bold; border-top: 2px solid #333;"
            ] * len(row)
            if row.name == n - 1 else [""] * len(row),
            axis=1,
        )
        .hide(axis="index")
    )
    display(styled)


# ── 1. Faixa Etária ────────────────────────────────────────────────────────────
c_faixa, _ = _carregar_e_preparar_casos(
    CSV_CASOS_FAIXA_ETARIA, "Casos - Faixa Etária",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
# Corrigir nomes e aplicar ordem decrescente de valor
c_faixa.index = [_corrigir_grafia(str(i)) for i in c_faixa.index]
c_faixa = pd.to_numeric(c_faixa, errors="coerce").fillna(0).astype(int)
c_faixa = c_faixa[c_faixa > 0].sort_values(ascending=False)

df_faixa = _build_tabela(c_faixa, "Faixa Etária")
_exibir_tabela(df_faixa, "Distribuição dos casos de câncer de mama por Faixa Etária em MG")


# ── 2. Escolaridade ───────────────────────────────────────────────────────────
c_esc, _ = _carregar_e_preparar_casos(
    CSV_CASOS_ESCOLARIDADE, "Casos - Escolaridade",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
c_esc.index = [_corrigir_grafia(str(i)) for i in c_esc.index]
c_esc = pd.to_numeric(c_esc, errors="coerce").fillna(0).astype(int)
c_esc = c_esc[c_esc > 0].sort_values(ascending=False)

df_esc = _build_tabela(c_esc, "Escolaridade")
_exibir_tabela(df_esc, "Distribuição dos casos de câncer de mama por Escolaridade em MG")


# ── 3. Raça/Cor ───────────────────────────────────────────────────────────────
c_raca, _ = _carregar_e_preparar_casos(
    CSV_CASOS_RACA_COR, "Casos - Raça/Cor",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
c_raca.index = [_corrigir_grafia(str(i)) for i in c_raca.index]
c_raca = pd.to_numeric(c_raca, errors="coerce").fillna(0).astype(int)
c_raca = c_raca[c_raca > 0].sort_values(ascending=False)

df_raca = _build_tabela(c_raca, "Raça/Cor")
_exibir_tabela(df_raca, "Distribuição dos casos de câncer de mama por Raça/Cor em MG")

ANOS_RANGE_CASOS      = (1940, 2025)

# ── 4. Ano de Diagnóstico ─────────────────────────────────────────────────────
df_ano_raw = aplicar_filtros(pd.read_csv(CSV_CASOS_ANO_DIAGNOSTICO))
ycols = [c for c in df_ano_raw.columns if str(c).isdigit() and ANOS_RANGE_CASOS[0] <= int(c) <= ANOS_RANGE_CASOS[1]]
c_ano = df_ano_raw[ycols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0)
c_ano.index = c_ano.index.astype(int)
c_ano = c_ano[c_ano > 0].sort_index(ascending=False)
c_ano.index = c_ano.index.astype(str)

df_ano = _build_tabela(c_ano, "Ano")
_exibir_tabela(df_ano, "Distribuição dos casos de câncer de mama por Ano de Diagnóstico em MG")

print(f"Total calculado: {c_ano.astype(int).sum():,}".replace(",", "."))